<a href="https://colab.research.google.com/github/sahitani/Gen-AI-Project/blob/main/dell_dispatch_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Import libraries and set random seed

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Setting random seed for reproducibility
# This is critical — same seed means same "random" data every time
np.random.seed(42)

print("Libraries imported ✓")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

Libraries imported ✓
Pandas version: 2.2.2
NumPy version:  2.0.2


In [2]:
# Cell 2 — Generate 10,000 realistic ticket records

N_RECORDS = 10000

# Helper lists for categorical features
issue_categories = ['Keyboard', 'Battery', 'Screen', 'Fan', 'Storage',
                    'Memory', 'Motherboard', 'Audio', 'Miscellaneous']
device_models = ['Latitude 5420', 'Latitude 7430', 'XPS 13', 'XPS 15',
                 'Precision 3560', 'Inspiron 5510', 'OptiPlex 7090']
warranty_types = ['Basic', 'ProSupport', 'ProSupport Plus']
regions = ['APJ', 'EMEA', 'AMER']

# Generate base data
data = {
    'ticket_id': [f'TKT-{str(i).zfill(6)}' for i in range(1, N_RECORDS + 1)],
    'issue_category': np.random.choice(issue_categories, N_RECORDS),
    'device_model': np.random.choice(device_models, N_RECORDS),
    'device_age_days': np.random.randint(30, 1500, N_RECORDS),
    'warranty_type': np.random.choice(warranty_types, N_RECORDS,
                                       p=[0.3, 0.5, 0.2]),
    'region': np.random.choice(regions, N_RECORDS, p=[0.25, 0.35, 0.40]),

    # Telemetry features — these are the new IoT insights
    'avg_cpu_temp_celsius': np.random.normal(65, 10, N_RECORDS).clip(40, 95),
    'max_cpu_temp_celsius': np.random.normal(82, 8, N_RECORDS).clip(55, 105),
    'battery_cycles': np.random.randint(50, 1200, N_RECORDS),
    'battery_health_pct': np.random.normal(85, 15, N_RECORDS).clip(20, 100),
    'disk_error_count': np.random.poisson(2, N_RECORDS),
    'memory_error_count': np.random.poisson(1, N_RECORDS),
    'boot_failure_count': np.random.poisson(0.5, N_RECORDS),
    'thermal_throttle_events': np.random.poisson(3, N_RECORDS),
    'avg_daily_uptime_hours': np.random.normal(8, 3, N_RECORDS).clip(1, 16),

    # Customer history
    'previous_ticket_count': np.random.poisson(2, N_RECORDS),
    'previous_false_dispatch_count': np.random.poisson(0.3, N_RECORDS),
    'days_since_last_ticket': np.random.randint(0, 365, N_RECORDS),

    # Agent interaction
    'customer_self_diagnosed': np.random.choice([0, 1], N_RECORDS,
                                                 p=[0.7, 0.3]),
    'symptom_severity_reported': np.random.randint(1, 6, N_RECORDS)
}

# Convert to DataFrame
df = pd.DataFrame(data)

print(f"Dataset created with {len(df)} rows and {len(df.columns)} columns")
print(f"\nFirst 3 rows:")
print(df.head(3))

Dataset created with 10000 rows and 20 columns

First 3 rows:
    ticket_id issue_category   device_model  device_age_days warranty_type  \
0  TKT-000001    Motherboard  Inspiron 5510              958         Basic   
1  TKT-000002            Fan         XPS 15              877    ProSupport   
2  TKT-000003          Audio         XPS 13              897    ProSupport   

  region  avg_cpu_temp_celsius  max_cpu_temp_celsius  battery_cycles  \
0   EMEA             59.971691             77.352436             762   
1   AMER             56.145344             75.661358             750   
2    APJ             67.718591             81.598918             940   

   battery_health_pct  disk_error_count  memory_error_count  \
0           84.067551                 0                   0   
1           59.210908                 3                   3   
2           85.825031                 1                   1   

   boot_failure_count  thermal_throttle_events  avg_daily_uptime_hours  \
0        

In [3]:
# Cell 3 — Generate the target label using business logic

# Initialise dispatch probability score for each row
dispatch_score = np.zeros(N_RECORDS)

# ============================================================
# FACTORS INCREASING TRUE DISPATCH LIKELIHOOD (hardware broken)
# ============================================================

# Telemetry degradation signals
dispatch_score += (df['max_cpu_temp_celsius'] - 80) * 0.05
dispatch_score += df['disk_error_count'] * 0.3
dispatch_score += df['memory_error_count'] * 0.4
dispatch_score += df['boot_failure_count'] * 0.5
dispatch_score += df['thermal_throttle_events'] * 0.1

# Device wear
dispatch_score += (df['battery_cycles'] / 500) * 0.4
dispatch_score += ((100 - df['battery_health_pct']) / 100) * 1.2

# Customer-reported severity
dispatch_score += (df['symptom_severity_reported'] - 3) * 0.4

# Device age
dispatch_score += (df['device_age_days'] / 1000) * 0.3

# ============================================================
# FACTORS DECREASING TRUE DISPATCH LIKELIHOOD (false alarm)
# ============================================================

# Customer self-diagnosis is often wrong
dispatch_score -= df['customer_self_diagnosed'] * 0.6

# Customer history of false dispatches
dispatch_score -= df['previous_false_dispatch_count'] * 0.5

# Add realistic noise — real life has randomness
noise = np.random.normal(0, 0.8, N_RECORDS)
dispatch_score += noise

# ============================================================
# CONVERT SCORE TO BINARY LABEL
# ============================================================

# Use threshold to create 80/20 split (realistic class imbalance)
threshold = np.percentile(dispatch_score, 20)

df['dispatch_outcome'] = np.where(dispatch_score > threshold,
                                   'TRUE_DISPATCH',
                                   'FALSE_DISPATCH')

# Check the distribution
print("Label distribution:")
print(df['dispatch_outcome'].value_counts())
print(f"\nFalse dispatch rate: {(df['dispatch_outcome'] == 'FALSE_DISPATCH').mean() * 100:.1f}%")

Label distribution:
dispatch_outcome
TRUE_DISPATCH     8000
FALSE_DISPATCH    2000
Name: count, dtype: int64

False dispatch rate: 20.0%


In [4]:
# Cell 4 — Introduce missing values and save dataset

# Simulate missing telemetry for ~5% of records (older devices)
missing_telemetry_indices = np.random.choice(
    df.index,
    size=int(N_RECORDS * 0.05),
    replace=False
)

telemetry_columns = ['avg_cpu_temp_celsius', 'max_cpu_temp_celsius',
                     'battery_cycles', 'battery_health_pct',
                     'disk_error_count', 'memory_error_count',
                     'boot_failure_count', 'thermal_throttle_events']

df.loc[missing_telemetry_indices, telemetry_columns] = np.nan

# Show missing value summary
print("Missing values per column:")
missing_summary = df.isnull().sum()
print(missing_summary[missing_summary > 0])

# Save to CSV
df.to_csv('dell_dispatch_data.csv', index=False)

print(f"\nDataset saved as dell_dispatch_data.csv")
print(f"Total rows:    {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nFinal dataset preview:")
print(df.head(3))

Missing values per column:
avg_cpu_temp_celsius       500
max_cpu_temp_celsius       500
battery_cycles             500
battery_health_pct         500
disk_error_count           500
memory_error_count         500
boot_failure_count         500
thermal_throttle_events    500
dtype: int64

Dataset saved as dell_dispatch_data.csv
Total rows:    10000
Total columns: 21

Final dataset preview:
    ticket_id issue_category   device_model  device_age_days warranty_type  \
0  TKT-000001    Motherboard  Inspiron 5510              958         Basic   
1  TKT-000002            Fan         XPS 15              877    ProSupport   
2  TKT-000003          Audio         XPS 13              897    ProSupport   

  region  avg_cpu_temp_celsius  max_cpu_temp_celsius  battery_cycles  \
0   EMEA             59.971691             77.352436           762.0   
1   AMER             56.145344             75.661358           750.0   
2    APJ             67.718591             81.598918           940.0   

   bat